In [ ]:
import pandas as pd

In [ ]:
dados = pd.read_csv('/content/dados2704.csv', index_col=False)
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21352 entries, 0 to 21351
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   TipoTomador                21352 non-null  int64  
 1   ReceitaBrutaMensal         21352 non-null  float64
 2   ValorTotaBem               21352 non-null  float64
 3   AnoEntrada                 21352 non-null  int64  
 4   AnoContratacao             21352 non-null  int64  
 5   MesContratacao             21352 non-null  int64  
 6   QuantidadeParcelas         21352 non-null  int64  
 7   TotalContratacoes          21352 non-null  int64  
 8   TotalContratacoesAtraso    21352 non-null  int64  
 9   ValorContratado            21352 non-null  float64
 10  AtividadeRenda             21352 non-null  int64  
 11  CodigoIBGE                 21352 non-null  int64  
 12  Alvo                       21352 non-null  int64  
 13  IBC                        21352 non-null  flo

In [ ]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import randint, uniform

# =========================================================
# Separação entre variáveis preditoras e variável alvo
# =========================================================

X = dados.drop(columns=['Alvo'])
y = dados['Alvo']

# =========================================================
# Tratamento de variáveis categóricas
# =========================================================

for coluna in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[coluna] = le.fit_transform(X[coluna].astype(str))

# =========================================================
# Separação treino e teste
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================================================
# Modelo base
# =========================================================

modelo = DecisionTreeClassifier(random_state=42)

# =========================================================
# Distribuições dos hiperparâmetros
# =========================================================

parametros = {
    'criterion': ['gini', 'entropy', 'log_loss'],

    'splitter': ['best', 'random'],

    'max_depth': randint(3, 30),

    'min_samples_split': randint(2, 50),

    'min_samples_leaf': randint(1, 30),

    'min_weight_fraction_leaf': uniform(0.0, 0.1),

    'max_features': [None, 'sqrt', 'log2'],

    'max_leaf_nodes': [None] + list(range(10, 201, 10)),

    'min_impurity_decrease': uniform(0.0, 0.05),

    'class_weight': [None, 'balanced'],

    'ccp_alpha': uniform(0.0, 0.05)
}

# =========================================================
# Configuração do Random Search
# =========================================================

random_search = RandomizedSearchCV(
    estimator=modelo,
    param_distributions=parametros,
    n_iter=100,                 # Quantidade de combinações testadas
    scoring='f1',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# =========================================================
# Treinamento
# =========================================================

random_search.fit(X_train, y_train)

# =========================================================
# Melhor configuração encontrada
# =========================================================

print("Melhores hiperparâmetros:")
print(random_search.best_params_)

print("\nMelhor F1-score médio:")
print(random_search.best_score_)

# =========================================================
# Avaliação no conjunto de teste
# =========================================================

melhor_modelo = random_search.best_estimator_

y_pred = melhor_modelo.predict(X_test)

print("\nF1-score no conjunto de teste:")
print(f1_score(y_test, y_pred))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred))

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Melhores hiperparâmetros:
{'ccp_alpha': np.float64(0.016685430556951094), 'class_weight': 'balanced', 'criterion': 'log_loss', 'max_depth': 24, 'max_features': None, 'max_leaf_nodes': 10, 'min_impurity_decrease': np.float64(0.03609993861334124), 'min_samples_leaf': 6, 'min_samples_split': 3, 'min_weight_fraction_leaf': np.float64(0.018182496720710064), 'splitter': 'best'}

Melhor F1-score médio:
0.6508437587743543

F1-score no conjunto de teste:
0.6440492476060191

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.88      0.63      0.73      2853
           1       0.53      0.83      0.64      1418

    accuracy                           0.70      4271
   macro avg       0.70      0.73      0.69      4271
weighted avg       0.76      0.70      0.70      4271


Matriz de confusão:
[[1793 1060]
 [ 241 1177]]


In [ ]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import randint, uniform

# Separação entre variáveis preditoras e variável alvo
X = dados.drop(columns=['Alvo'])
y = dados['Alvo']

# Tratamento simples de variáveis categóricas
for coluna in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[coluna] = le.fit_transform(X[coluna].astype(str))

# Separação treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Modelo base
modelo = RandomForestClassifier(random_state=42)

# Distribuições dos hiperparâmetros
parametros = {
    'n_estimators': randint(100, 1000),
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [None] + list(range(3, 31)),
    'min_samples_split': randint(2, 50),
    'min_samples_leaf': randint(1, 30),
    'min_weight_fraction_leaf': uniform(0.0, 0.1),
    'max_features': ['sqrt', 'log2', None],
    'max_leaf_nodes': [None] + list(range(10, 201, 10)),
    'min_impurity_decrease': uniform(0.0, 0.05),
    'bootstrap': [True, False],
    'class_weight': [None, 'balanced', 'balanced_subsample'],
    'ccp_alpha': uniform(0.0, 0.05),
    'max_samples': uniform(0.5, 0.5)
}

# Random Search usando F1-score
random_search = RandomizedSearchCV(
    estimator=modelo,
    param_distributions=parametros,
    n_iter=100,
    scoring='f1',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Treinamento
random_search.fit(X_train, y_train)

# Melhores hiperparâmetros encontrados
print("Melhores hiperparâmetros:")
print(random_search.best_params_)

print("\nMelhor F1-score médio na validação cruzada:")
print(random_search.best_score_)

# Avaliação no conjunto de teste
melhor_modelo = random_search.best_estimator_
y_pred = melhor_modelo.predict(X_test)

print("\nF1-score no conjunto de teste:")
print(f1_score(y_test, y_pred))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred))

Fitting 5 folds for each of 100 candidates, totalling 500 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
235 fits failed out of a total of 500.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
235 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/ensemble/_forest.py", line 431, in fit
    raise ValueError(
ValueErr

Melhores hiperparâmetros:
{'bootstrap': True, 'ccp_alpha': np.float64(0.022524962598477152), 'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 29, 'max_features': 'log2', 'max_leaf_nodes': 170, 'max_samples': np.float64(0.692708251269958), 'min_impurity_decrease': np.float64(0.0007983126110107097), 'min_samples_leaf': 2, 'min_samples_split': 21, 'min_weight_fraction_leaf': np.float64(0.024102546602601173), 'n_estimators': 554}

Melhor F1-score médio na validação cruzada:
0.6631384392111032

F1-score no conjunto de teste:
0.6582278481012658

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.88      0.68      0.77      2853
           1       0.56      0.81      0.66      1418

    accuracy                           0.72      4271
   macro avg       0.72      0.74      0.71      4271
weighted avg       0.77      0.72      0.73      4271


Matriz de confusão:
[[1939  914]
 [ 274 1144]]


In [ ]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from scipy.stats import randint, uniform, loguniform

# Separação entre variáveis preditoras e variável alvo
X = dados.drop(columns=['Alvo'])
y = dados['Alvo']

# Tratamento simples de variáveis categóricas
for coluna in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[coluna] = le.fit_transform(X[coluna].astype(str))

# Separação treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Modelo base
modelo = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=1
)

# Distribuições dos hiperparâmetros
parametros = {
    # Quantidade de árvores
    'n_estimators': randint(100, 1000),

    # Taxa de aprendizado
    'learning_rate': loguniform(0.001, 0.3),

    # Profundidade das árvores
    'max_depth': randint(2, 15),

    # Mínima redução de perda para nova divisão
    'gamma': uniform(0, 10),

    # Soma mínima dos pesos das instâncias em um nó filho
    'min_child_weight': randint(1, 20),

    # Amostragem de registros
    'subsample': uniform(0.5, 0.5),

    # Amostragem de colunas por árvore
    'colsample_bytree': uniform(0.5, 0.5),

    # Amostragem de colunas por nível
    'colsample_bylevel': uniform(0.5, 0.5),

    # Amostragem de colunas por nó
    'colsample_bynode': uniform(0.5, 0.5),

    # Regularização L1
    'reg_alpha': loguniform(1e-5, 10),

    # Regularização L2
    'reg_lambda': loguniform(1e-5, 10),

    # Peso da classe positiva
    'scale_pos_weight': uniform(1, 20),

    # Estratégia de crescimento
    'grow_policy': ['depthwise', 'lossguide'],

    # Número máximo de folhas
    'max_leaves': randint(0, 100),

    # Método de construção das árvores
    'tree_method': ['hist'],

    # Número máximo de bins
    'max_bin': randint(64, 512)
}

# Random Search
random_search = RandomizedSearchCV(
    estimator=modelo,
    param_distributions=parametros,
    n_iter=100,
    scoring='f1',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Treinamento
random_search.fit(X_train, y_train)

# Melhores hiperparâmetros
print("Melhores hiperparâmetros:")
print(random_search.best_params_)

print("\nMelhor F1-score médio na validação cruzada:")
print(random_search.best_score_)

# Avaliação no conjunto de teste
melhor_modelo = random_search.best_estimator_

y_pred = melhor_modelo.predict(X_test)

print("\nF1-score no conjunto de teste:")
print(f1_score(y_test, y_pred))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred))

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Melhores hiperparâmetros:
{'colsample_bylevel': np.float64(0.7995146823886632), 'colsample_bynode': np.float64(0.9133994139045392), 'colsample_bytree': np.float64(0.9795373972820138), 'gamma': np.float64(3.425242571785434), 'grow_policy': 'lossguide', 'learning_rate': np.float64(0.009839149612911786), 'max_bin': 366, 'max_depth': 11, 'max_leaves': 45, 'min_child_weight': 11, 'n_estimators': 495, 'reg_alpha': np.float64(2.958792117688386), 'reg_lambda': np.float64(6.834349194024739e-05), 'scale_pos_weight': np.float64(3.01589206031773), 'subsample': np.float64(0.628007765926831), 'tree_method': 'hist'}

Melhor F1-score médio na validação cruzada:
0.7372588654959246

F1-score no conjunto de teste:
0.7348571428571429

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.94      0.72      0.82      2853
           1       0.62      0.91      0.73      1418

    accuracy        